# NB04 — Data Cleaning & Merge

## Purpose

This notebook merges the outputs from NB01 (Hospital Characteristics), NB02 (DRG/CMI), and NB03 (Payment Data & Severity) into a single master hospital dataset. Each row represents one hospital (identified by CCN), with all available metrics joined via left join from our primary hospital universe (NB01).

## Cell 1: Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Define project root and file paths
PROJECT_ROOT = Path('..').resolve().parent
print(f"Project Root: {PROJECT_ROOT}")

# Input file paths (relative to this notebook's directory)
nb01_path = PROJECT_ROOT / 'data' / 'outputs' / 'nb01_hospital_characteristics' / 'hospital_characteristics.csv'
nb02_path = PROJECT_ROOT / 'data' / 'outputs' / 'nb02_drg_cmi' / 'hospital_drg_data.csv'
nb03_payment_path = PROJECT_ROOT / 'data' / 'outputs' / 'nb03_payment_data' / 'hospital_payment_data.csv'
nb03_severity_path = PROJECT_ROOT / 'data' / 'outputs' / 'nb03_payment_data' / 'hospital_severity_distribution.csv'

# Output directory and file path
output_dir = PROJECT_ROOT / 'data' / 'outputs' / 'nb04_merged'
output_path = output_dir / 'master_hospital_dataset.csv'

# Create output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

# Verify all input files exist
input_files = {
    'NB01 Hospital Characteristics': nb01_path,
    'NB02 DRG/CMI': nb02_path,
    'NB03 Payment Data': nb03_payment_path,
    'NB03 Severity Distribution': nb03_severity_path
}

for name, path in input_files.items():
    exists = path.exists()
    print(f"{name}: {'✓' if exists else '✗'} {path}")

Project Root: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap
Output directory: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb04_merged
NB01 Hospital Characteristics: ✓ /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb01_hospital_characteristics/hospital_characteristics.csv
NB02 DRG/CMI: ✓ /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb02_drg_cmi/hospital_drg_data.csv
NB03 Payment Data: ✓ /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb03_

## Cell 2: Load All Input Files

In [2]:
# Load all four input files with CCN as string
nb01_df = pd.read_csv(nb01_path, dtype={'ccn': str})
nb02_df = pd.read_csv(nb02_path, dtype={'ccn': str})
nb03_payment_df = pd.read_csv(nb03_payment_path, dtype={'ccn': str})
nb03_severity_df = pd.read_csv(nb03_severity_path, dtype={'ccn': str})

# Standardize CCN format (6-digit zero-padded)
nb01_df['ccn'] = nb01_df['ccn'].str.zfill(6)
nb02_df['ccn'] = nb02_df['ccn'].str.zfill(6)
nb03_payment_df['ccn'] = nb03_payment_df['ccn'].str.zfill(6)
nb03_severity_df['ccn'] = nb03_severity_df['ccn'].str.zfill(6)

# Print row counts and basic info
print("=" * 60)
print("LOADED DATA SUMMARY")
print("=" * 60)
print(f"NB01 Hospital Characteristics: {len(nb01_df):,} rows")
print(f"NB02 DRG/CMI Data: {len(nb02_df):,} rows")
print(f"NB03 Payment Data: {len(nb03_payment_df):,} rows")
print(f"NB03 Severity Distribution: {len(nb03_severity_df):,} rows")
print("\n" + "=" * 60)

# Display column info
print("\nNB01 Columns:")
print(list(nb01_df.columns))
print("\nNB02 Columns:")
print(list(nb02_df.columns))
print("\nNB03 Payment Columns:")
print(list(nb03_payment_df.columns))
print("\nNB03 Severity Columns:")
print(list(nb03_severity_df.columns))

LOADED DATA SUMMARY
NB01 Hospital Characteristics: 3,280 rows
NB02 DRG/CMI Data: 3,151 rows
NB03 Payment Data: 2,945 rows
NB03 Severity Distribution: 2,871 rows


NB01 Columns:
['ccn', 'hospital_name', 'city', 'state', 'zip_code', 'county', 'beds', 'bed_size_tier', 'ownership', 'ownership_category', 'is_teaching', 'is_urban', 'census_region', 'peer_group', 'overall_rating']

NB02 Columns:
['ccn', 'hospital_name', 'cmi', 'discharges', 'wage_index', 'beds', 'teaching_pct', 'dsh_pct']

NB03 Payment Columns:
['ccn', 'discharges', 'total_charges', 'total_payments', 'total_medicare_payments', 'distinct_drgs', 'hospital_name', 'state', 'charge_to_payment_ratio', 'avg_payment_per_discharge']

NB03 Severity Columns:
['ccn', 'with_CC', 'with_MCC', 'without_CC', 'total_cc_discharges', 'pct_without_CC', 'pct_with_CC', 'pct_with_MCC']


## Cell 3: Merge Strategy

### Overview

We perform **left joins** from our primary hospital universe (NB01) to secondary sources:

1. **NB01** (n=`X` hospitals) serves as the base universe from the POS file
2. **NB02** (DRG/CMI) joins on CCN → adds CMI, wage index, and resident-to-bed ratio
3. **NB03 Payment** joins on CCN → adds charge, payment, and discharge metrics
4. **NB03 Severity** joins on CCN → adds case mix severity distribution

### Column Handling for Duplicates

When multiple sources have the same column (e.g., `hospital_name`, `beds`, `discharges`):
- **Prefer NB01** (POS-based): hospital_name, beds (staff beds from licensing)
- **Keep NB02 separately**: rename discharges → `medicare_discharges_impact`, beds → `staffed_beds_cms` (if present)
- **Keep NB03 separately**: rename discharges → `medicare_discharges_puf`, use in calculations

This allows us to track disagreements and understand data source differences.

## Cell 4: Merge NB01 ← NB02

In [3]:
print("\n" + "=" * 60)
print("MERGE STEP 1: NB01 ← NB02 (DRG/CMI)")
print("=" * 60)

# Rename NB02 columns to avoid conflicts
# Keep hospital_name_nb02 and beds_nb02 for comparison
nb02_rename = {
    'discharges': 'medicare_discharges_impact',
    'beds': 'staffed_beds_cms',
    'hospital_name': 'hospital_name_nb02',
    'teaching_pct': 'resident_to_bed_ratio'
}
nb02_df = nb02_df.rename(columns=nb02_rename)

# Left join NB01 to NB02 on CCN
merged_df = nb01_df.merge(
    nb02_df[['ccn', 'cmi', 'wage_index', 'resident_to_bed_ratio', 'dsh_pct', 
             'medicare_discharges_impact', 'staffed_beds_cms', 'hospital_name_nb02']],
    on='ccn',
    how='left'
)

# Match statistics
match_count_nb02 = merged_df['cmi'].notna().sum()
print(f"Hospitals with NB02 match: {match_count_nb02:,} / {len(merged_df):,} ({100*match_count_nb02/len(merged_df):.1f}%)")
print(f"Merged shape: {merged_df.shape}")

# Show summary of key metrics
print(f"\nNB02 Key Metrics:")
print(f"  CMI: mean={merged_df['cmi'].mean():.3f}, min={merged_df['cmi'].min():.3f}, max={merged_df['cmi'].max():.3f}")
print(f"  Resident-to-Bed Ratio: mean={merged_df['resident_to_bed_ratio'].mean():.3f}")
print(f"  Wage Index: mean={merged_df['wage_index'].mean():.3f}")


MERGE STEP 1: NB01 ← NB02 (DRG/CMI)
Hospitals with NB02 match: 3,060 / 3,280 (93.3%)
Merged shape: (3280, 22)

NB02 Key Metrics:
  CMI: mean=1.768, min=0.654, max=4.884
  Resident-to-Bed Ratio: mean=0.091
  Wage Index: mean=1.040


## Cell 5: Merge ← NB03 Payment Data

In [4]:
print("\n" + "=" * 60)
print("MERGE STEP 2: merged ← NB03 Payment Data")
print("=" * 60)

# Rename NB03 payment columns
nb03_payment_rename = {
    'discharges': 'medicare_discharges_puf',
    'hospital_name': 'hospital_name_nb03',
    'state': 'state_nb03'
}
nb03_payment_df = nb03_payment_df.rename(columns=nb03_payment_rename)

# Left join to payment data
merged_df = merged_df.merge(
    nb03_payment_df[['ccn', 'total_charges', 'total_payments', 'total_medicare_payments',
                     'distinct_drgs', 'charge_to_payment_ratio', 'avg_payment_per_discharge',
                     'medicare_discharges_puf', 'hospital_name_nb03', 'state_nb03']],
    on='ccn',
    how='left'
)

# Match statistics
match_count_nb03 = merged_df['total_payments'].notna().sum()
print(f"Hospitals with NB03 payment match: {match_count_nb03:,} / {len(merged_df):,} ({100*match_count_nb03/len(merged_df):.1f}%)")
print(f"Merged shape: {merged_df.shape}")

# Show summary of key payment metrics
print(f"\nNB03 Payment Key Metrics:")
print(f"  Total Payments: mean=${merged_df['total_payments'].mean():,.0f}, median=${merged_df['total_payments'].median():,.0f}")
print(f"  Total Medicare Payments: mean=${merged_df['total_medicare_payments'].mean():,.0f}")
print(f"  Charge-to-Payment Ratio: mean={merged_df['charge_to_payment_ratio'].mean():.3f}")
print(f"  Distinct DRGs: mean={merged_df['distinct_drgs'].mean():.1f}")


MERGE STEP 2: merged ← NB03 Payment Data
Hospitals with NB03 payment match: 2,880 / 3,280 (87.8%)
Merged shape: (3280, 31)

NB03 Payment Key Metrics:
  Total Payments: mean=$30,632,445, median=$12,106,142
  Total Medicare Payments: mean=$25,426,140
  Charge-to-Payment Ratio: mean=4.473
  Distinct DRGs: mean=50.7


## Cell 6: Merge ← NB03 Severity Distribution

In [5]:
print("\n" + "=" * 60)
print("MERGE STEP 3: merged ← NB03 Severity Distribution")
print("=" * 60)

# Left join to severity distribution
merged_df = merged_df.merge(
    nb03_severity_df[['ccn', 'with_CC', 'with_MCC', 'without_CC', 
                       'total_cc_discharges', 'pct_without_CC', 'pct_with_CC', 'pct_with_MCC']],
    on='ccn',
    how='left'
)

# Match statistics
match_count_severity = merged_df['pct_with_CC'].notna().sum()
print(f"Hospitals with severity distribution match: {match_count_severity:,} / {len(merged_df):,} ({100*match_count_severity/len(merged_df):.1f}%)")
print(f"Final merged shape: {merged_df.shape}")

# Show summary of severity metrics
print(f"\nNB03 Severity Key Metrics:")
print(f"  % Without CC/MCC: mean={merged_df['pct_without_CC'].mean():.1f}%")
print(f"  % With CC: mean={merged_df['pct_with_CC'].mean():.1f}%")
print(f"  % With MCC: mean={merged_df['pct_with_MCC'].mean():.1f}%")


MERGE STEP 3: merged ← NB03 Severity Distribution
Hospitals with severity distribution match: 2,817 / 3,280 (85.9%)
Final merged shape: (3280, 38)

NB03 Severity Key Metrics:
  % Without CC/MCC: mean=5.3%
  % With CC: mean=20.4%
  % With MCC: mean=74.4%


## Cell 7: Data Quality Checks

We now assess the quality and completeness of the merged dataset:

1. **Missing Values**: Identify which columns have gaps and their extent
2. **Duplicate CCNs**: Verify each hospital appears exactly once
3. **Data Completeness**: Classify hospitals by which data sources they matched (full vs. partial)

## Cell 8: Missing Values & Data Completeness

In [6]:
print("\n" + "=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

# Check for duplicate CCNs
duplicate_ccns = merged_df['ccn'].duplicated().sum()
if duplicate_ccns == 0:
    print("✓ No duplicate CCNs found")
else:
    print(f"✗ WARNING: {duplicate_ccns} duplicate CCNs found")

# Summary of missing values
print("\nMissing Values by Source:")
print("-" * 60)

# Key columns from each source
nb02_cols = ['cmi', 'wage_index', 'resident_to_bed_ratio']
nb03_payment_cols = ['total_charges', 'total_payments', 'charge_to_payment_ratio']
nb03_severity_cols = ['pct_with_CC', 'pct_with_MCC']

nb02_missing = merged_df[nb02_cols].isna().any(axis=1).sum()
nb03_payment_missing = merged_df[nb03_payment_cols].isna().any(axis=1).sum()
nb03_severity_missing = merged_df[nb03_severity_cols].isna().any(axis=1).sum()

print(f"NB02 (CMI/Wage): {nb02_missing:,} hospitals missing data ({100*nb02_missing/len(merged_df):.1f}%)")
print(f"NB03 Payment: {nb03_payment_missing:,} hospitals missing data ({100*nb03_payment_missing/len(merged_df):.1f}%)")
print(f"NB03 Severity: {nb03_severity_missing:,} hospitals missing data ({100*nb03_severity_missing/len(merged_df):.1f}%)")

# Create data completeness flag using vectorized pandas operations (not row-level apply)
merged_df['data_completeness'] = np.where(
    merged_df['cmi'].notna() &
    merged_df['total_payments'].notna() &
    merged_df['pct_with_CC'].notna(),
    'full', 'partial'
)

completeness_counts = merged_df['data_completeness'].value_counts()
print("\nData Completeness:")
for status, count in completeness_counts.items():
    pct = 100 * count / len(merged_df)
    print(f"  {status.upper()}: {count:,} hospitals ({pct:.1f}%)")

print("\nDetailed Missing Value Counts:")
missing_summary = merged_df.isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
if len(missing_summary) > 0:
    for col, count in missing_summary.items():
        pct = 100 * count / len(merged_df)
        print(f"  {col}: {count:,} ({pct:.1f}%)")
else:
    print("  No missing values in key columns")


DATA QUALITY CHECKS
✓ No duplicate CCNs found

Missing Values by Source:
------------------------------------------------------------
NB02 (CMI/Wage): 220 hospitals missing data (6.7%)
NB03 Payment: 400 hospitals missing data (12.2%)
NB03 Severity: 463 hospitals missing data (14.1%)

Data Completeness:
  FULL: 2,815 hospitals (85.8%)
  PARTIAL: 465 hospitals (14.2%)

Detailed Missing Value Counts:
  pct_with_MCC: 463 (14.1%)
  pct_with_CC: 463 (14.1%)
  pct_without_CC: 463 (14.1%)
  total_cc_discharges: 463 (14.1%)
  without_CC: 463 (14.1%)
  with_MCC: 463 (14.1%)
  with_CC: 463 (14.1%)
  hospital_name_nb03: 400 (12.2%)
  medicare_discharges_puf: 400 (12.2%)
  total_medicare_payments: 400 (12.2%)
  total_payments: 400 (12.2%)
  total_charges: 400 (12.2%)
  avg_payment_per_discharge: 400 (12.2%)
  distinct_drgs: 400 (12.2%)
  charge_to_payment_ratio: 400 (12.2%)
  state_nb03: 400 (12.2%)
  hospital_name_nb02: 221 (6.7%)
  cmi: 220 (6.7%)
  wage_index: 220 (6.7%)
  resident_to_bed_ratio

## Cell 9: Data Type Standardization & Column Cleanup

In [7]:
print("\n" + "=" * 60)
print("DATA TYPE STANDARDIZATION & CLEANUP")
print("=" * 60)

# Ensure numeric columns are properly typed
numeric_columns = [
    'beds', 'staffed_beds_cms', 'cmi', 'wage_index', 'resident_to_bed_ratio', 'dsh_pct',
    'medicare_discharges_impact', 'medicare_discharges_puf',
    'total_charges', 'total_payments', 'total_medicare_payments',
    'distinct_drgs', 'charge_to_payment_ratio', 'avg_payment_per_discharge',
    'with_CC', 'with_MCC', 'without_CC', 'total_cc_discharges',
    'pct_without_CC', 'pct_with_CC', 'pct_with_MCC'
]

for col in numeric_columns:
    if col in merged_df.columns:
        merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

# Drop redundant/helper columns from data source comparisons
# (Keep hospital_name_nb02 and hospital_name_nb03 for validation if needed)
cols_to_drop = ['hospital_name_nb02', 'hospital_name_nb03', 'state_nb03']
merged_df = merged_df.drop(columns=[col for col in cols_to_drop if col in merged_df.columns])

# Define final column order
final_columns = [
    # Identifiers
    'ccn', 'hospital_name', 'city', 'state',
    # Hospital characteristics
    'zip_code', 'county', 'beds', 'bed_size_tier', 'ownership', 'ownership_category',
    'is_teaching', 'is_urban', 'census_region', 'peer_group', 'overall_rating',
    # CMS/DRG metrics
    'cmi', 'wage_index', 'resident_to_bed_ratio', 'dsh_pct',
    'staffed_beds_cms', 'medicare_discharges_impact',
    # Payment data
    'total_charges', 'total_payments', 'total_medicare_payments',
    'avg_payment_per_discharge', 'charge_to_payment_ratio',
    'distinct_drgs', 'medicare_discharges_puf',
    # Severity distribution
    'with_CC', 'with_MCC', 'without_CC', 'total_cc_discharges',
    'pct_without_CC', 'pct_with_CC', 'pct_with_MCC',
    # Data quality flag
    'data_completeness'
]

# Reorder columns (only keep those that exist)
existing_cols = [col for col in final_columns if col in merged_df.columns]
merged_df = merged_df[existing_cols]

print(f"\nFinal dataset shape: {merged_df.shape}")
print(f"Final columns: {len(merged_df.columns)}")
print("\nData Types:")
print(merged_df.dtypes)


DATA TYPE STANDARDIZATION & CLEANUP

Final dataset shape: (3280, 36)
Final columns: 36

Data Types:
ccn                            object
hospital_name                  object
city                           object
state                          object
zip_code                        int64
county                         object
beds                          float64
bed_size_tier                  object
ownership                      object
ownership_category             object
is_teaching                      bool
is_urban                         bool
census_region                  object
peer_group                     object
overall_rating                 object
cmi                           float64
wage_index                    float64
resident_to_bed_ratio         float64
dsh_pct                       float64
staffed_beds_cms              float64
medicare_discharges_impact    float64
total_charges                 float64
total_payments                float64
total_medicare_payments  

## Cell 10: Summary Statistics & Sample Rows

In [8]:
print("\n" + "=" * 60)
print("MASTER DATASET SUMMARY STATISTICS")
print("=" * 60)

print(f"\nTotal Hospitals: {len(merged_df):,}")
print(f"Total Columns: {len(merged_df.columns)}")

print("\n" + "-" * 60)
print("HOSPITAL CHARACTERISTICS")
print("-" * 60)
print(f"Teaching Hospitals: {merged_df['is_teaching'].sum():,} ({100*merged_df['is_teaching'].mean():.1f}%)")
print(f"Urban Hospitals: {merged_df['is_urban'].sum():,} ({100*merged_df['is_urban'].mean():.1f}%)")
print(f"\nMean Beds: {merged_df['beds'].mean():.1f}")
print(f"Median Beds: {merged_df['beds'].median():.1f}")
print(f"Max Beds: {merged_df['beds'].max():.0f}")

print("\n" + "-" * 60)
print("CASE MIX INDICATORS")
print("-" * 60)
print(f"Mean CMI: {merged_df['cmi'].mean():.3f}")
print(f"CMI Range: {merged_df['cmi'].min():.3f} to {merged_df['cmi'].max():.3f}")
print(f"Mean Wage Index: {merged_df['wage_index'].mean():.3f}")
print(f"Mean DSH Pct: {merged_df['dsh_pct'].mean():.1f}%")

print("\n" + "-" * 60)
print("PAYMENT METRICS")
print("-" * 60)
print(f"Total Medicare Payments (all hospitals): ${merged_df['total_medicare_payments'].sum():,.0f}")
print(f"Mean Payment per Hospital: ${merged_df['total_medicare_payments'].mean():,.0f}")
print(f"Mean Charge-to-Payment Ratio: {merged_df['charge_to_payment_ratio'].mean():.3f}")
print(f"Mean DRGs per Hospital: {merged_df['distinct_drgs'].mean():.1f}")

print("\n" + "-" * 60)
print("SEVERITY DISTRIBUTION")
print("-" * 60)
print(f"Mean % Without CC/MCC: {merged_df['pct_without_CC'].mean():.1f}%")
print(f"Mean % With CC: {merged_df['pct_with_CC'].mean():.1f}%")
print(f"Mean % With MCC: {merged_df['pct_with_MCC'].mean():.1f}%")

print("\n" + "-" * 60)
print("SAMPLE ROWS (First 5 Hospitals)")
print("-" * 60)
display_cols = ['ccn', 'hospital_name', 'state', 'beds', 'cmi', 'total_payments', 'pct_with_CC', 'data_completeness']
print(merged_df[display_cols].head(5).to_string())

print("\n" + "-" * 60)
print("OWNERSHIP DISTRIBUTION")
print("-" * 60)
if 'ownership_category' in merged_df.columns:
    ownership_dist = merged_df['ownership_category'].value_counts()
    for ownership, count in ownership_dist.items():
        pct = 100 * count / len(merged_df)
        print(f"  {ownership}: {count:,} ({pct:.1f}%)")


MASTER DATASET SUMMARY STATISTICS

Total Hospitals: 3,280
Total Columns: 36

------------------------------------------------------------
HOSPITAL CHARACTERISTICS
------------------------------------------------------------
Teaching Hospitals: 1,167 (35.6%)
Urban Hospitals: 2,568 (78.3%)

Mean Beds: 268.2
Median Beds: 185.0
Max Beds: 3289

------------------------------------------------------------
CASE MIX INDICATORS
------------------------------------------------------------
Mean CMI: 1.768
CMI Range: 0.654 to 4.884
Mean Wage Index: 1.040
Mean DSH Pct: 0.3%

------------------------------------------------------------
PAYMENT METRICS
------------------------------------------------------------
Total Medicare Payments (all hospitals): $73,227,284,543
Mean Payment per Hospital: $25,426,140
Mean Charge-to-Payment Ratio: 4.473
Mean DRGs per Hospital: 50.7

------------------------------------------------------------
SEVERITY DISTRIBUTION
-----------------------------------------------

## Cell 11: Save Master Dataset

In [9]:
print("\n" + "=" * 60)
print("SAVING MASTER DATASET")
print("=" * 60)

# Save to output path
merged_df.to_csv(output_path, index=False)

print(f"✓ Master dataset saved to: {output_path}")
print(f"  File size: {output_path.stat().st_size / (1024**2):.2f} MB")
print(f"  Shape: {merged_df.shape[0]:,} rows × {merged_df.shape[1]} columns")
print(f"  Format: CSV (index=False)")

print(f"\n" + "=" * 60)
print("✓ MERGE PROCESS COMPLETE")
print("=" * 60)
print(f"\nThe master hospital dataset combines:")
print(f"  • NB01: Hospital Characteristics (POS file)")
print(f"  • NB02: Case Mix & DRG Metrics (CMS MDE)")
print(f"  • NB03: Payment Data & Severity Distribution (Claims)")
print(f"\nOutput: {output_path}")
print(f"Hospitals: {len(merged_df):,}")
print(f"Full Data: {(merged_df['data_completeness'] == 'full').sum():,} ({100*(merged_df['data_completeness'] == 'full').mean():.1f}%)")


SAVING MASTER DATASET


✓ Master dataset saved to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb04_merged/master_hospital_dataset.csv
  File size: 1.09 MB
  Shape: 3,280 rows × 36 columns
  Format: CSV (index=False)

✓ MERGE PROCESS COMPLETE

The master hospital dataset combines:
  • NB01: Hospital Characteristics (POS file)
  • NB02: Case Mix & DRG Metrics (CMS MDE)
  • NB03: Payment Data & Severity Distribution (Claims)

Output: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb04_merged/master_hospital_dataset.csv
Hospitals: 3,280
Full Data: 2,815 (85.8%)
